# Hello world Transformers 🤗

Exploring Hugging Face Transformers for various NLP tasks.


## Setup

Installing required packages:


In [ ]:
%pip install transformers pandas torch


## 1. Input Data

Defining the sample text we'll use for all tasks.


In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""


### Understanding Pipelines

**What is a pipeline?** A pipeline is a high-level abstraction that automatically handles three steps:
1. **Preprocessing** - tokenizing the input text
2. **Model inference** - passing inputs to the model
3. **Postprocessing** - converting model outputs to human-readable labels

**What does it abstract away?** It hides all the complexity of tokenization, model loading, tensor operations, and output formatting. You just pass in text and get results.

**Other tasks available:** Besides text-classification, pipelines support many tasks like:
- `ner` (Named Entity Recognition)
- `question-answering`
- `summarization`
- `text-generation`
- `translation`
- `zero-shot-classification`
- `sentiment-analysis`
- And many more...

**No model specified?** If you don't specify a model, the pipeline defaults to a pre-trained model for that task. For text classification, it uses `distilbert-base-uncased-finetuned-sst-2-english`.

**How to specify a specific model?** You can pass the `model` parameter:
```python
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
```

## 2. Text Classification

Using a pipeline to classify sentiment. Pipelines handle preprocessing, model inference, and postprocessing automatically.


In [ ]:
from transformers import pipeline
import pandas as pd

classifier = pipeline("text-classification")
outputs = classifier(text)
pd.DataFrame(outputs)


### Deep Dive

**Default Model:** `distilbert-base-uncased-finetuned-sst-2-english`

**Dataset:** Fine-tuned on SST-2 (Stanford Sentiment Treebank) dataset, which contains movie reviews labeled as positive or negative.

**What kind of text does it work best with?** It works best with movie reviews and similar opinionated text, since that's what it was trained on. It may be less accurate on other domains like technical documentation or news articles.

**Score:** Represents the confidence/probability of the predicted label. The score range is **0 to 1**, where values closer to 1 indicate higher confidence. Higher score = more confident prediction.

**Emotion Model:** For more granular classification, you can use models like `j-hartmann/emotion-english-distilroberta-base` which classify emotions (joy, anger, sadness, fear, etc.) rather than just positive/negative sentiment.


## 3. Named Entity Recognition (NER)

Identifying entities (organizations, locations, persons) in the text. Using `aggregation_strategy="simple"` to group subword tokens back into whole words.


In [ ]:
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)


### NER Analysis

**Aggregation Strategy:** `aggregation_strategy="simple"` groups sub-word tokens (like "Mega" and "##tron") back into whole words ("Megatron") for cleaner output. Without this, you'd see each subword token separately.

**Entity Types:**
- **ORG** - Organization (e.g., Amazon)
- **LOC** - Location (e.g., Germany)
- **PER** - Person (e.g., Bumblebee)
- **MISC** - Miscellaneous (e.g., Optimus Prime, Megatron)

**## Tokens:** The `##` prefix indicates a subword token created by the tokenizer. BERT uses WordPiece tokenization, where words are split into subwords. The `##` means this chunk is attached to the previous token (not the start of a new word).

**Splitting:** "Megatron" and "Decepticons" were split because they're fictional names not in the model's standard vocabulary. The model was trained on CoNLL-2003, which focuses on real-world news data, so fictional character names aren't in its vocabulary.

**CoNLL-2003 Dataset:** The CoNLL-2003 dataset is a standard benchmark for NER, containing English news articles from Reuters. It includes four entity types: PER (person), LOC (location), ORG (organization), and MISC (miscellaneous). The model `dbmdz/bert-large-cased-finetuned-conll03-english` was fine-tuned on this dataset, which is why it performs well on news-style text but struggles with fictional names.


## 4. Question Answering

Extractive QA - finds the answer span directly in the text.


In [ ]:
reader = pipeline("question-answering")

question = "What does the customer want?"
outputs = reader(question=question, context=text)
pd.DataFrame([outputs])


### QA Systems

**Type:** This is **Extractive QA**. It extracts the answer directly from the provided text rather than generating a new sentence. The model finds the span of text that best answers the question.

**Indices:** `start` and `end` represent the character positions in the text where the answer is located. This allows you to extract the exact answer substring from the original text. They're important because they let you locate and highlight the exact answer in the original document.

**SQuAD:** The Stanford Question Answering Dataset. The default model `distilbert-base-cased-distilled-squad` was trained on this dataset, which contains questions and answers based on Wikipedia articles.

**Questions it CANNOT answer:** This model cannot answer questions that require:
- **Reasoning beyond the text:** "Why did the customer choose Amazon?" (requires inference)
- **Information not in the text:** "What is Amazon's return policy?" (not mentioned)
- **Synthesis across multiple parts:** "Compare the customer's tone at the beginning vs end" (requires analysis)
- **Future predictions:** "What will Amazon do next?" (not extractable)

It fails because extractive QA can only highlight existing text spans - it cannot reason, infer, or generate new information.

**Generative QA:** Unlike extractive QA, generative models (like T5 or GPT) create new text to answer questions rather than highlighting a span of existing text. They can synthesize information and provide answers even when the exact words aren't in the context. Examples include `google/flan-t5-base` or `facebook/bart-large`.


## 5. Summarization

Generating a short summary. The default model is `sshleifer/distilbart-cnn-12-6` (abstractive, based on BART, trained on CNN/DailyMail).


### Summarization Details

**Extractive vs. Abstractive:**
- **Extractive:** Selects existing sentences from the text to form a summary
- **Abstractive:** Generates new sentences to capture the meaning (like a human would write)

**Default Model:** `sshleifer/distilbart-cnn-12-6` - This is an abstractive model based on BART architecture, trained on the CNN/DailyMail dataset.

**Parameters:**
- `max_length`: Limits the maximum length of the summary (in tokens). The model will try to generate up to this many tokens.
- `min_length`: Sets the minimum length of the summary (in tokens). Ensures the summary isn't too short.
- **What happens if min_length > max_length?** This would cause an error or unexpected behavior, as the model cannot generate a summary that's both longer than max_length and shorter than min_length. Always ensure min_length < max_length.

**clean_up_tokenization_spaces:** Removes artifacts like spaces before punctuation marks, which improves readability of the output. This is especially useful because tokenizers sometimes add extra spaces that look unnatural.

**Why is summarization challenging?** It requires understanding meaning, identifying key information, and generating coherent text - much more complex than classification which just assigns a label.

**Different Summarization Models:**
- **For short texts:** `facebook/bart-large-cnn` - Optimized for news articles, trained on CNN/DailyMail
- **For longer documents:** `google/pegasus-xsum` - Can handle longer contexts, trained on XSum dataset
- **Comparison:** BART uses encoder-decoder architecture, while Pegasus uses a different pre-training objective. Both are abstractive models but optimized for different document lengths.


In [ ]:
summarizer = pipeline("summarization")
outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])


## 6. Translation

Translating from English to German using MarianMT architecture.


### Machine Translation Details

**Model Architecture:** `Helsinki-NLP/opus-mt-en-de` uses the **MarianMT** architecture, which is a neural machine translation framework based on the Transformer architecture. It's an encoder-decoder model optimized for translation.

**OPUS:** Open Parallel Corpus - a large collection of parallel texts used as training data for machine translation models.

**MT:** Machine Translation

**Finding English to French models:** You can search the Hugging Face Hub for translation models:
1. `Helsinki-NLP/opus-mt-en-fr` - Bilingual MarianMT model for English->French
2. `facebook/mbart-large-50` - Multilingual model that supports English->French among many other pairs
3. Use `pipeline("translation_en_to_fr")` or `pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")`

**Bilingual vs. Multilingual:**
- **Bilingual models:** Handle one specific language pair (e.g., English -> German). They're specialized and often more accurate for that specific pair. Advantages: Better quality for the specific pair. Disadvantages: Need separate models for each language pair.
- **Multilingual models:** Can handle many languages (e.g., mBART, mT5). More flexible but may be less accurate for specific pairs compared to specialized bilingual models. Advantages: One model for many languages. Disadvantages: May sacrifice some accuracy.

**Task specification:** When we use `pipeline("translation_en_to_de")`, it automatically selects an appropriate model for that language pair. Alternatively, we can explicitly specify the model with `model="Helsinki-NLP/opus-mt-en-de"`. The task name is a shortcut that maps to a default model.

**Sacremoses:** A library for Moses-compatible tokenizer/detokenizer, often required for MarianMT models to properly handle text preprocessing and postprocessing. It handles sentence segmentation, tokenization, and truecasing - important steps for translation quality.

**Multilingual Models:**
- **mBART (facebook/mbart-large-50):** Supports 50 languages and can translate between any pair
- **M2M100 (facebook/m2m100_418M):** Supports 100 languages with direct translation between any pair
- **NLLB (facebook/nllb-200-3.3B):** Supports 200 languages, including many low-resource languages
These models are trained on massive multilingual corpora and can handle translation between languages they've seen, even if not explicitly trained on that specific pair.


In [ ]:
translator = pipeline("translation_en_to_de", model="Helsinki-NLP/opus-mt-en-de")
outputs = translator(text, clean_up_tokenization_spaces=True, min_length=100)
print(outputs[0]['translation_text'])


## 7. Text Generation

Using GPT-2 to generate a customer service response. Setting seed for reproducibility.


In [ ]:
from transformers import set_seed

set_seed(42)

generator = pipeline("text-generation")

response = "Dear Bumblebee, I am sorry to hear that your order was mixed up."
prompt = text + "\n\nCustomer service response:\n" + response

outputs = generator(prompt, max_length=200)
print(outputs[0]['generated_text'])


### Text Generation Details

**Default Model:** `gpt2` (OpenAI GPT-2). It's a decoder-only transformer that performs **autoregressive generation** - predicting the next word based on all previous words, one token at a time.

**Architecture:** GPT-2 uses a **decoder-only** architecture (not encoder-decoder or encoder-only). It only has decoder layers from the Transformer architecture.

**Parameters:** The base GPT-2 model has **124 million parameters**. There are also larger variants: GPT-2 Medium (355M), GPT-2 Large (774M), and GPT-2 XL (1.5B).

**Generation Type:** It performs **autoregressive generation** - generating one token at a time, where each new token depends on all previously generated tokens.

**Seed:** `set_seed(42)` ensures that the random aspects of generation (like sampling) produce the same result every time you run the code. Without it, you'd get different outputs each time due to randomness in the sampling process. This is important for reproducibility and debugging.

**Parameters:**
- **`temperature`:** Controls randomness in generation
  - Higher temperature (e.g., 1.0-1.5) = more creative/random outputs
  - Lower temperature (e.g., 0.1-0.5) = more deterministic, focused outputs
  - Default is usually 1.0
- **`top_k`:** Limits the model to sampling from the top k most likely next words. Reduces randomness by ignoring low-probability tokens. For example, top_k=50 means only consider the 50 most likely words.
- **`do_sample`:** 
  - `True` = picks the next word based on probability distribution (more diverse)
  - `False` = always picks the most likely word (greedy search, more deterministic)
- **`max_length`:** Maximum total length of the generated sequence (input + generated tokens)

**Truncation:** The warning occurs because the input prompt plus the generated tokens might exceed the model's maximum context length (usually 1024 tokens for GPT-2). The model will truncate the input if it's too long, which means some of your prompt might be cut off.

**pad_token_id and eos_token_id:** GPT-2 doesn't have a padding token by default. When `pad_token_id` is set to `eos_token_id` (end-of-sequence token), it means the model will use the end-of-sequence token for padding. This is necessary because:
- During batch processing, sequences need to be padded to the same length
- GPT-2 needs a token to represent padding
- Using eos_token_id prevents the model from treating padding as actual content

**Trade-offs between model size and generation quality:**
- **Larger models (GPT-2 XL, GPT-3):** Better quality, more coherent text, better understanding of context, but require more memory, slower inference, and higher computational cost
- **Smaller models (GPT-2 base):** Faster inference, less memory, lower cost, but may produce less coherent text, struggle with long contexts, and have less nuanced understanding
- Generally, more parameters = better quality but at the cost of efficiency
